# CHB Quickstart — hardness fingerprints & regimes in 3 lines

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Walid10010/CHB/blob/main/examples/quickstart.ipynb)
[![ICML 2026](https://img.shields.io/badge/ICML-2026-blue.svg)](https://openreview.net/pdf?id=9zXUOLxbcL)

**CHB** (Clustering Hardness Benchmark) embeds a labeled dataset into an interpretable
hardness space **h(D) = (S; C; T)** — separation, cohesion, topology — and assigns one of
three **regimes**:

| Regime | Meaning |
|---|---|
| **A** | Separability collapse — local evidence is unreliable, all method families fail similarly |
| **B** | Topology mismatch — separable, but loops/voids/anisotropy break "blob" assumptions |
| **C** | Scale heterogeneity — broadly recoverable; difficulty comes from non-uniform scales |

Paper: *CHB: A Diagnostic Toolkit for Hardness-Aware Clustering Evaluation* (ICML 2026).
Labels are used **for diagnosis only** — never to fit clustering.

In [1]:
try:
    import chb
except ImportError:
    %pip install -q git+https://github.com/Walid10010/CHB.git  # after the PyPI release: %pip install -q chb-clustering
    import chb

print("CHB version:", chb.__version__)

CHB version: 1.0.0


## 1. Three lines on a real dataset

`sklearn`'s digits (8×8 handwritten digits, 1797×64) — no download needed. Runtime ≈ 20–30 s.

In [2]:
from sklearn.datasets import load_digits

X, y = load_digits(return_X_y=True)
fingerprint, regime = chb.compute_fingerprint(X, y)

print("regime:", regime)
fingerprint

regime: C


{'S1': 0.05035573122529644,
 'S2': 0.07908031156482032,
 'S3': 1.2552179996209454,
 'C1': 0.2788796919142803,
 'C2': 15.89059876142255,
 'T1': 697.5127205865175,
 'T2': 26.731107676784116,
 'T3': 3.8275623184239533}

### Reading the fingerprint

| Axis | Descriptors | Orientation |
|---|---|---|
| **Separation** | S1 overlap, S2 hubness (↑ harder), S3 margin (↓ harder) | is there local evidence for membership? |
| **Cohesion** | C1 density complexity, C2 elongation (↑ harder) | do clusters share one scale/shape? |
| **Topology** | T1 (PH0), T2 (PH1 loops), T3 (PH2 voids) (↑ harder) | chains, loops, voids beyond "blobs"? |

The rich result object exposes the separability **gate** (2-of-3 rule via
`SEPF = median(S1−0.5, S2−0.33, 1.0−S3)`), the blob-calibrated **topology evidence**
`T_evid = Σ log(1+Tᵢ)` with cutoff **15.0**, and the full per-block report:

In [3]:
res = chb.compute_fingerprint(X, y)

g = res.gate
print(f"Gate: SEPF={g['SEPF']:+.3f} -> {'FAIL' if g['gate_fails'] else 'pass'} "
      f"({g['n_strict_failures']}/3 strict violations)")
print(f"T_evid = {res.t_evid:.2f}  (tau_top = 15.0)")
print(f"Regime = {res.regime}")

Gate: SEPF=-0.255 -> pass (0/3 strict violations)
T_evid = 11.45  (tau_top = 15.0)
Regime = C


**Digits lands in Regime C** — the gate passes cleanly and topology evidence stays
below the blob-null cutoff. Regime C means *broadly recoverable*: plain KMeans reaches
NMI ≈ 0.67 here, close to the paper's Regime-C median (≈ 0.70).

## 2. Same task, different representation → different regime

Digits is essentially a low-resolution MNIST. Raw **MNIST (28×28)** sits in **Regime B**
(fingerprint below = paper Table 15). The 8×8 representation acts like an aggressive
dimensionality reduction: it **simplifies topology** (T1: 12103 → 698, T2: 179 → 27,
T3: 24.5 → 3.8) while keeping separability — exactly the *mismatch-correction* mechanism
(B→C) the paper identifies for representation learning. `chb.assign_regime` re-derives
regimes from stored fingerprints without recomputing anything:

In [4]:
mnist_raw = {"S1": 0.081, "S2": 0.107, "S3": 1.096, "C1": 0.196,
             "C2": 67.236, "T1": 12102.879, "T2": 179.404, "T3": 24.541}

print("MNIST raw (784-d):", chb.assign_regime(mnist_raw), "| T1 =", mnist_raw["T1"])
print("digits    (64-d): ", regime, "| T1 =", round(fingerprint["T1"], 1))

MNIST raw (784-d): B | T1 = 12102.879
digits    (64-d):  C | T1 = 697.5


## 3. Negative control: separability collapse (Regime A)

Heavily overlapping Gaussians — local neighborhoods stop carrying label evidence,
the gate fails, and CHB predicts near-universal method failure (Regime A):

In [5]:
import numpy as np

rng = np.random.default_rng(2)
centers = rng.uniform(-3, 3, size=(4, 10))
X_bad = np.vstack([c + 4.0 * rng.standard_normal((200, 10)) for c in centers])
y_bad = np.repeat(np.arange(4), 200)

res_bad = chb.compute_fingerprint(X_bad, y_bad)
g = res_bad.gate
print(f"Gate: SEPF={g['SEPF']:+.3f} -> {'FAIL' if g['gate_fails'] else 'pass'} "
      f"({g['n_strict_failures']}/3 strict violations)")
print("Regime =", res_bad.regime)

Gate: SEPF=+0.160 -> FAIL (3/3 strict violations)
Regime = A


## Where to go next

* **CLI:** `chb both --input your.csv --label-col target` writes a combined JSON report
  (fingerprint + gate + T_evid + regime); `chb chb --report dir/` annotates existing reports.
* **Full report:** `res.report` holds every per-cluster descriptor and all secondary
  (`sec_*`) diagnostics.
* **Repo & docs:** <https://github.com/Walid10010/CHB> · **Paper:**
  [OpenReview](https://openreview.net/pdf?id=9zXUOLxbcL)

If you use CHB, please cite the ICML 2026 paper (see `CITATION.cff` / repo README).